In [15]:
import os
import requests
from PIL import Image
from io import BytesIO

import pandas as pd
import matplotlib.pyplot as plt

import numpy as np

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from matplotlib.animation import FuncAnimation
import matplotlib.colors as mcolors
from datetime import datetime, timedelta

In [16]:
# NASA GIBS WMS endpoint
BASE_URL = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"

# Output folders
IMAGE_DIR = "modis_sst_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# Image size for downloaded maps
WIDTH = 800
HEIGHT = 800

In [26]:
MODIS_LAYERS = {
    "Aqua_Night": "MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily",
    "Aqua_Day": "MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily",
}

In [27]:
REGIONS = {
    "East Coast": {
        "bbox": "-90,-50,15,65",
        "description": "Western Atlantic along the U.S. East Coast"
    }
}

REGIONS

{'East Coast': {'bbox': '-90,-50,15,65',
  'description': 'Western Atlantic along the U.S. East Coast'}}

In [28]:
def make_filename(region, satellite, time_of_day, date):
    """
    Creates a clean filename for a MODIS SST image.
    """
    region_clean = region.lower().replace(" ", "_")
    satellite_clean = satellite.lower()
    time_clean = time_of_day.lower()
    date_clean = date.replace("-", "_")
    
    return f"{region_clean}_{satellite_clean}_{time_clean}_{date_clean}.png"


def download_modis_sst_image(region, satellite, time_of_day, date):
    """
    Downloads one MODIS SST image from NASA GIBS WMS API.

    Parameters
    ----------
    region : str
        Region name from REGIONS.
    satellite : str
        Either "Aqua" or "Terra".
    time_of_day : str
        Either "Day" or "Night".
    date : str
        Date in YYYY-MM-DD format.

    Returns
    -------
    dict
        Metadata about the downloaded image.
    """
    
    layer_key = f"{satellite}_{time_of_day}"
    layer_name = MODIS_LAYERS[layer_key]
    bbox = REGIONS[region]["bbox"]
    
    filename = make_filename(region, satellite, time_of_day, date)
    filepath = os.path.join(IMAGE_DIR, filename)
    
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer_name,
        "STYLES": "",
        "CRS": "EPSG:4326",
        "BBOX": bbox,
        "WIDTH": str(WIDTH),
        "HEIGHT": str(HEIGHT),
        "FORMAT": "image/png",
        "TRANSPARENT": "false",
        "TIME": date
    }
    
    response = requests.get(BASE_URL, params=params)
    content_type = response.headers.get("Content-Type", "")
    
    print(date, region, satellite, time_of_day, response.status_code, content_type)
    
    if "image" in content_type:
        with open(filepath, "wb") as f:
            f.write(response.content)
            
        status = "success"
        error_message = ""
    else:
        filepath = None
        status = "failed"
        error_message = response.text[:500]
        print("NASA returned an error:")
        print(error_message)
    
    return {
        "date": date,
        "year": int(date[:4]),
        "month": int(date[5:7]),
        "region": region,
        "bbox": bbox,
        "satellite": satellite,
        "time_of_day": time_of_day,
        "layer": layer_name,
        "image_path": filepath,
        "status": status,
        "error_message": error_message
    }

In [31]:
all_metadata = []

years = [2012]
months = [10]
days = [25,26,28,29,30,31]
regions = list(REGIONS.keys())
satellites = ["Aqua", "Aqua"]
times_of_day = ['Day','Night']


for region in regions:
    for satellite in satellites:
        for time_of_day in times_of_day:
            for month in months:
                for day in days:
                    date = f"2012-10-{day}"
                    
                    meta = download_modis_sst_image(
                        region=region,
                        satellite=satellite,
                        time_of_day = time_of_day,
                        date=date
                    )
                    
                    all_metadata.append(meta)

full_metadata_df = pd.DataFrame(all_metadata)
full_metadata_df.head()


2012-10-25 East Coast Aqua Day 200 text/xml; charset=UTF-8
NASA returned an error:
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.3.0/exceptions_1_3_0.xsd">
<ServiceException code="LayerNotDefined">
msWMSLoadGetMapParams(): WMS server error. Unable to access -- invalid LAYER(s)
</ServiceException>
</ServiceExceptionReport>

2012-10-26 East Coast Aqua Day 200 text/xml; charset=UTF-8
NASA returned an error:
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.3.0/exceptions_1_3_0.xsd">
<ServiceException code="LayerNotDefined">
msWMSLoadGetMapParams(): WMS server err

,date,year,month,region,bbox,satellite,time_of_day,layer,image_path,status,error_message
0,2012-10-25,2012,10,East Coast,"-90,-50,15,65",Aqua,Day,MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily,NaN,failed,"<?xml version='1.0' encoding=""UTF-8"" standalon..."
1,2012-10-26,2012,10,East Coast,"-90,-50,15,65",Aqua,Day,MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily,NaN,failed,"<?xml version='1.0' encoding=""UTF-8"" standalon..."
2,2012-10-28,2012,10,East Coast,"-90,-50,15,65",Aqua,Day,MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily,NaN,failed,"<?xml version='1.0' encoding=""UTF-8"" standalon..."
3,2012-10-29,2012,10,East Coast,"-90,-50,15,65",Aqua,Day,MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily,NaN,failed,"<?xml version='1.0' encoding=""UTF-8"" standalon..."
4,2012-10-30,2012,10,East Coast,"-90,-50,15,65",Aqua,Day,MODIS_Aqua_L3_SST_MidIR_9km_Day_Daily,NaN,failed,"<?xml version='1.0' encoding=""UTF-8"" standalon..."


In [32]:
successful_metadata_df = full_metadata_df[full_metadata_df["status"] == "success"].copy()

successful_metadata_df.to_csv("modis_sst_metadata.csv", index=False)

print("Saved modis_sst_metadata.csv")
print("Number of successful images:", len(successful_metadata_df))
successful_metadata_df.head()

Saved modis_sst_metadata.csv
Number of successful images: 12


,date,year,month,region,bbox,satellite,time_of_day,layer,image_path,status,error_message
6,2012-10-25,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,
7,2012-10-26,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,
8,2012-10-28,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,
9,2012-10-29,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,
10,2012-10-30,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,


In [33]:


df = pd.read_csv("modis_sst_metadata.csv")

df["image_path"] = df["image_path"].astype(str)

df["image_path"] = df["image_path"].apply(
    lambda path: path.replace("\\", "/").split("modis_sst_images/")[-1]
)

df["image_path"] = "modis_sst_images/" + df["image_path"]

df.to_csv("modis_sst_metadata.csv", index=False)

df.head()

,date,year,month,region,bbox,satellite,time_of_day,layer,image_path,status,error_message
0,2012-10-25,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,NaN
1,2012-10-26,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,NaN
2,2012-10-28,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,NaN
3,2012-10-29,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,NaN
4,2012-10-30,2012,10,East Coast,"-90,-50,15,65",Aqua,Night,MODIS_Aqua_L3_SST_MidIR_9km_Night_Daily,modis_sst_images/east_coast_aqua_night_2012_10...,success,NaN
